# 3 · Train and predict

Train on the same Dynamic World images and labels as notebooks 0 and 1. Build a catalog of those files, prepare dated chips, and train the model.

Then use the KHDTK imagery saved by notebook 2 for inference only, joining chip predictions back into dated land-cover maps.


In [ ]:
import warnings
from pathlib import Path

import lightning as L
import torch
from huggingface_hub import snapshot_download
from torch.utils.data import DataLoader, Dataset

import geosave as gs

L.seed_everything(0)
warnings.filterwarnings("ignore", ".*does not have many workers.*")

root = Path(
    snapshot_download(
        "fatmur/dynamicworld-300", repo_type="dataset", local_dir="data/dynamic_world"
    )
)

## Load the classes

Read `classes.yaml` from the downloaded dataset. These constants define the class IDs, colours and ignored pixels for every step below.


In [ ]:
import yaml

classes = yaml.safe_load((root / "classes.yaml").read_text())
IGNORE_INDEX = classes["ignore_index"]
CLASS_MAP = classes["class_map"]
COLOR_MAP = classes["color_map"]
NUM_CLASSES = len(CLASS_MAP)
CLASS_MAP

## Define the Dataset

Keep the path-based interface from the earlier examples: image paths, optional label paths, and a transform. `__init__` builds the catalog and chip index from metadata. `__getitem__` reads one chip and calls `preprocess()` to produce its tensors.


In [ ]:
def preprocess(tree):
    """Mask, scale and convert one chip into model tensors."""
    image = tree["sentinel-2-l2a"].dataset
    image = image.gs.mask_and_scale().gs.to_tensor(dtype="float32")
    image = image.nan_to_num()
    if "label" not in tree.children:
        return image

    label = tree["label"].dataset["label"]
    label = label.gs.mask_and_scale().gs.to_tensor(dtype="float32")
    return image, label.nan_to_num(IGNORE_INDEX).long()


class DynamicWorld(Dataset):
    def __init__(self, images, labels=None, *, transform=None, overlap=0):
        self.images = images
        self.labels = [] if labels is None else labels
        self.transform = transform
        self.catalog = self._create_catalog()

        samples = self.catalog.gs.sample()
        self.frames = samples.gs.frame(
            windows={"sentinel-2-l2a": {"slots": ["0D"]}},
            reference="label" if self.labels else None,
        )
        self.chips = self._chip(overlap)

    def _create_catalog(self):
        items = []
        for path in self.images:
            items.extend(
                gs.stac.create_items(
                    {"image": path}, collection="sentinel-2-l2a", id_format=Path(path).stem
                )
            )
        for path in self.labels:
            items.extend(
                gs.stac.create_items(
                    {"label": path}, collection="label", id_format=Path(path).stem
                )
            )
        return gs.stac.create_catalog(items)

    def _chip(self, overlap):
        # Tiles span different CRSs; keep each frame on its image's native grid.
        tables = []
        for index, row in self.frames.iterrows():
            tree = row.gs.to_xarray(collections=["sentinel-2-l2a"])
            grid = tree["sentinel-2-l2a"].dataset.gs.geobox
            tables.append(
                self.frames.loc[[index]].gs.chip(128, target=grid, overlap=overlap)
            )
        return gs.GeoSamples.concat(tables).reset_index(drop=True)

    def __len__(self):
        return len(self.chips)

    def __getitem__(self, index):
        row = self.chips.iloc[index]
        tree = row.gs.to_xarray(target=row.gs.geobox).isel(time=0)
        if self.transform is not None:
            return self.transform(tree)
        return tree

## Set up the DataModule

The DataModule creates each split's Dataset in `setup()` and supplies DataLoaders to Trainer. Catalogs and chip indexes stay inside those Datasets. KHDTK is not used here.

The split checks let us inspect the training Dataset now and reuse it when Trainer calls `setup()` again.


In [ ]:
class DynamicWorldDataModule(L.LightningDataModule):
    def __init__(self, root="data/dynamic_world", batch_size=8):
        super().__init__()
        self.root = Path(root)
        self.batch_size = batch_size
        self.train_dataset = None
        self.val_dataset = None
        self.test_dataset = None

    def setup(self, stage=None):
        if stage in (None, "fit") and self.train_dataset is None:
            self.train_dataset = self._create_dataset("train")
        if stage in (None, "fit", "validate") and self.val_dataset is None:
            self.val_dataset = self._create_dataset("val")
        if stage in (None, "test") and self.test_dataset is None:
            self.test_dataset = self._create_dataset("test")

    def _create_dataset(self, split):
        images = sorted((self.root / split / "images").glob("*.tif"))
        labels = sorted((self.root / split / "labels").glob("*.tif"))
        return DynamicWorld(images, labels, transform=preprocess)

    def train_dataloader(self):
        return DataLoader(self.train_dataset, batch_size=self.batch_size, shuffle=True)

    def val_dataloader(self):
        return DataLoader(self.val_dataset, batch_size=self.batch_size)

    def test_dataloader(self):
        return DataLoader(self.test_dataset, batch_size=self.batch_size)

Set up the training and validation Datasets so we can inspect them before training. Trainer also calls `setup()` when needed.


In [ ]:
data = DynamicWorldDataModule(root)
data.setup("fit")

## Inspect the training catalog

The Dataset has already cataloged its image and label paths during initialization. Inspect that catalog and the preparation steps it uses, without reading entire rasters.


In [ ]:
catalog = data.train_dataset.catalog
catalog

Read one row as an xarray Dataset. Opening it reads metadata; pixel values are loaded when needed.


In [ ]:
row = catalog.iloc[0]
raster = row.gs.to_xarray()
raster

## Group scenes by place

`sample()` puts scenes with the same footprint in one row. It keeps every date and each collection's own grid.


In [ ]:
samples = catalog.gs.sample()
samples

Read a sample as a DataTree: one Dataset per collection.


In [ ]:
row = samples.iloc[0]
tree = row.gs.to_xarray()
tree

## Choose the dates

Walk through the preparation steps for one training sample. Use the label's timestamp as the reference. `"0D"` selects the image from that day. The label is included automatically.

Each slot produces one time step. For example, `["-2W", "-1W"]` requests the two weeks before the reference. Add `"reduce": "first"` to choose the first scene in each week.


In [ ]:
frames = samples.iloc[:1].gs.frame(
    windows={"sentinel-2-l2a": {"slots": ["0D"]}},
    reference="label",
)
frames

Read the frame to see the selected image and label.


In [ ]:
row = frames.iloc[0]
tree = row.gs.to_xarray()
tree

## Cut 128 × 128 pixel chips

Use this sample's image grid for its image and label. The original training tiles span different coordinate systems, so each sample keeps its own grid. `chip()` records pixel windows without loading their pixels.


In [ ]:
grid = tree["sentinel-2-l2a"].dataset.gs.geobox
chips = frames.gs.chip(128, target=grid)
chips

Read one chip. `target=row.gs.geobox` puts the image and label on the chip's grid so their pixels line up.


In [ ]:
row = chips.iloc[0]
tree = row.gs.to_xarray(target=row.gs.geobox)
tree

## View the chip

Plotting loads its pixels. Tiled TIFFs let the reader load only the storage blocks needed for this chip.


In [ ]:
tree.gs.plot(cols=2)

The prepared training and validation splits contain these numbers of chips:


In [ ]:
len(data.train_dataset), len(data.val_dataset)

## Build the model

The model reads four bands and predicts a class for each pixel. The dataset's class map gives the number of output channels, ordered by class ID. Its ignore index marks pixels excluded from training and evaluation.

The constructor sets the band count, chip size and model width.


In [ ]:
CLASS_MAP

In [ ]:
import kornia.augmentation as K
from torch import nn
from geosave.model import build_model
from torchmetrics import MetricCollection
from torchmetrics.classification import MulticlassAccuracy, MulticlassJaccardIndex


class Segmenter(L.LightningModule):
    def __init__(self, num_classes, ignore_index, in_channels=4, input_size=128, width=16):
        super().__init__()
        self.model = build_model(
            {
                "encoder": {
                    "name": "small_cnn",
                    "init_args": {
                        "in_channels": in_channels,
                        "input_size": input_size,
                        "width": width,
                    },
                },
                "decoder": {"name": "unet"},
                "head": {
                    "name": "segmentation",
                    "init_args": {"num_classes": num_classes},
                },
            }
        )
        self.augment = K.AugmentationSequential(
            K.RandomHorizontalFlip(),
            K.RandomVerticalFlip(),
            data_keys=["image", "mask"],
            keepdim=True,
        )
        self.val_metrics = MetricCollection(
            {
                "miou": MulticlassJaccardIndex(
                    num_classes=num_classes, ignore_index=ignore_index
                ),
                "accuracy": MulticlassAccuracy(
                    num_classes=num_classes, ignore_index=ignore_index, average="micro"
                ),
            },
            prefix="val_",
        )
        self.test_metrics = self.val_metrics.clone(prefix="test_")
        self.criterion = nn.CrossEntropyLoss(ignore_index=ignore_index)

    def forward(self, image):
        return self.model(image=image)

    def training_step(self, batch, batch_index):
        image, label = self.augment(*batch)  # (B, C, H, W), (B, H, W)
        logits = self(image)  # (B, classes, H, W)
        loss = self.criterion(logits, label)
        self.log("train_loss", loss)
        return loss

    def validation_step(self, batch, batch_index):
        image, label = batch  # (B, C, H, W), (B, H, W)
        logits = self(image)  # (B, classes, H, W)
        loss = self.criterion(logits, label)
        self.log("val_loss", loss)
        self.val_metrics(logits, label)
        self.log_dict(self.val_metrics)

    def test_step(self, batch, batch_index):
        image, label = batch
        logits = self(image)
        self.log("test_loss", self.criterion(logits, label))
        self.test_metrics(logits, label)
        self.log_dict(self.test_metrics)

    def configure_optimizers(self):
        return torch.optim.AdamW(self.parameters(), lr=1e-2)


module = Segmenter(
    num_classes=NUM_CLASSES,
    ignore_index=IGNORE_INDEX,
    in_channels=4,
    input_size=128,
    width=16,
)
module

## Train

Run six short epochs. Logs and checkpoints go to `lightning_logs/03_train/`. Running again resumes the last checkpoint.


In [ ]:
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.loggers import TensorBoardLogger

logger = TensorBoardLogger("lightning_logs", name="03_train", version=0)
checkpoint = ModelCheckpoint(save_last=True)

trainer = L.Trainer(
    max_epochs=6,
    limit_train_batches=6,
    limit_val_batches=2,
    log_every_n_steps=1,
    logger=logger,
    callbacks=[checkpoint],
)
trainer.fit(module, datamodule=data, ckpt_path="last")

Evaluate the held-out test split. This small dataset demonstrates the workflow; it does not measure regional accuracy.


In [ ]:
trainer.test(module, datamodule=data)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir lightning_logs

## Predict over the forest

KHDTK is used only for inference. Pass the image paths saved by notebook 2 to the same Dataset, without labels. It prepares dated chips on the KHDTK images' own grid.


In [ ]:
region = gs.open_vector("khdtk.geojson")
image_paths = sorted(Path("data/khdtk/sentinel-2-l2a").glob("*.tif"))

Without a label collection, each image timestamp supplies a reference. The same Dataset prepares frames and chips on the KHDTK imagery grid.


In [ ]:
dataset = DynamicWorld(image_paths, transform=preprocess)
dataset.frames

Inspect the prepared chip table.


In [ ]:
dataset.chips

Pass the Dataset to a DataLoader. Its `frames` and `chips` retain the layout needed to reconstruct predictions.


In [ ]:
prediction_loader = DataLoader(dataset, batch_size=8)

## Predict each chip

Run the model without gradients. Keep the class scores so we can merge chips before choosing a class.


In [ ]:
module.eval()
batches = []
with torch.no_grad():
    for images in prediction_loader:
        logits = module(images.to(module.device))
        batches.append(logits.cpu())

## Postprocess predictions

Keep reconstruction and output metadata together in `postprocess()`. It matches scores to chip rows, merges them on each frame's grid, and chooses the winning class ID. Pixels without valid scores keep the dataset's ignore index.

The returned label raster carries dates, nodata, class names and colours from `classes.yaml`.


In [ ]:
def postprocess(logits, frames, chips, *, taper=None):
    """Merge chip scores into dated land-cover maps using the dataset's classes."""
    # Output channels follow ascending class IDs from classes.yaml.
    class_ids = sorted(CLASS_MAP)
    names = [CLASS_MAP[class_id] for class_id in class_ids]

    # Match each chip output to its row, then merge scores before choosing classes.
    outputs = {}
    for index, values in zip(chips.index, logits, strict=True):
        outputs[index] = values
    merged = frames.gs.merge(chips, outputs, bands=names, taper=taper)

    maps = []
    for index, raster in merged.items():
        # Select class IDs rather than channel positions; uncovered pixels stay nodata.
        scores = raster.to_dataarray("class_id").assign_coords(class_id=class_ids)
        labels = scores.idxmax("class_id", fill_value=IGNORE_INDEX).astype("uint8")
        # Restore the date of the frame these chips came from.
        time = frames.loc[index, "start_datetime"]
        maps.append(labels.assign_coords(time=time))

    # Assemble dated maps and attach the metadata needed for plotting and export.
    cover = gs.transform.temporal.concat(maps).rename("label")
    cover = cover.gs.write_nodata(IGNORE_INDEX)
    return cover.gs.write_legend(CLASS_MAP, COLOR_MAP)

Merge scores before choosing classes so overlapping chips can contribute to the same pixel.


In [ ]:
logits = torch.cat(batches).numpy()
forest_cover = postprocess(logits, dataset.frames, dataset.chips)

Plot the dated land-cover maps with their class legend.


In [ ]:
forest_cover.gs.plot()

## Save the maps

Write one TIFF per date to `data/khdtk_pred/land_cover/`, then save their catalog as `data/khdtk_pred/catalog.parquet`.

The catalog uses relative file paths. Keep it with the raster folder when moving the results.


In [ ]:
folder = Path("data/khdtk_pred")

cover_scenes = forest_cover.gs.to_cog(
    folder / "land_cover/{time:%Y%m%d}.tif", overwrite=True
)
cover_collection = gs.stac.create_collection(
    "land_cover",
    description="Land-cover predictions for individual acquisition days from the tutorial model",
)
cover_items = gs.stac.create_items(
    cover_scenes, collection=cover_collection, id_format="{start:%Y%m%d}"
)
cover_catalog = gs.stac.create_catalog(cover_items, collections=[cover_collection])
cover_catalog.gs.to_geoparquet(folder / "catalog.parquet", overwrite=True)
cover_catalog

In [ ]:
forest_cover.gs.explore(overlay={"KHDTK": region}, basemap="Imagery")

## What happened

We turned catalog rows into dated chips, trained a model, predicted without labels, and joined the predictions into maps. The maps and their catalog are ready to reopen.


## Your turn

Try these small experiments. Open a hint if you need help getting started.

### 1 · Which date has the most trees?

Count how many pixels the model called trees in each map. Which date has the largest count?

<details><summary>Hint</summary>

Trees have class ID `1`. Compare `forest_cover` with `1`, then add up the matching pixels for each date.

```python
(forest_cover == 1).sum(["y", "x"]).to_series()
```
</details>

In [ ]:
# Your code here

### 2 · What if we only count inside the forest?

The maps also include land outside the forest outline. Count the tree pixels again, this time only inside the outline. How do the counts compare with the first exercise?

<details><summary>Hint</summary>

Make a mask from the forest outline using the prediction map's grid. Keep the pixels inside it, then count the trees as before.

```python
inside = region.gs.rasterize(forest_cover)
(forest_cover.where(inside) == 1).sum(["y", "x"]).to_series()
```
</details>

In [ ]:
# Your code here

### 3 · Would a bigger model help?

Train a slightly bigger version of the model using the same images. Does its validation accuracy improve?

<details><summary>Hint</summary>

Create `Segmenter(num_classes=NUM_CLASSES, ignore_index=IGNORE_INDEX, width=32)` instead of using the default width of `16`. Keep the other settings the same so you can compare the two runs.

Use a new name such as `03_wider` in `TensorBoardLogger`, then create a new `Segmenter` and Trainer. This starts a fresh run instead of resuming the earlier checkpoint.
</details>

In [ ]:
# Your code here

### 4 · Can you make the joins less visible?

Look for lines where neighbouring prediction patches meet. Try overlapping chips, then merge them with a tapered window.

<details><summary>Hint</summary>

Create the KHDTK Dataset with overlap:

```python
dataset = DynamicWorld(image_paths, transform=preprocess, overlap=32)
prediction_loader = DataLoader(dataset, batch_size=8)
```

Repeat inference with the trained model, then call:

```python
forest_cover = postprocess(
    logits, dataset.frames, dataset.chips, taper="hann"
)
```

This gives more weight to the middle of each chip. Plot the same date before and after the change. You do not need to retrain the model.
</details>


In [ ]:
# Your code here